# 02 — Streaming Bronze (usage_events)

Ingesta de `landing/usage_events_stream/*.jsonl` hacia **Bronze** con Spark Structured Streaming.

**Requisitos MVP:** schema explícito, `withWatermark`, dedupe por `event_id`, late data, checkpointing.

> **Nota watermark:** el landing estático cubre ~60 días. Para replay completo usamos `60 days` (configurable con `STREAMING_WATERMARK`). En producción near-real-time el diseño usa `10 minutes`.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pyspark
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/cloud-provider-analytics")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_ROOT, LANDING, BRONZE, CHECKPOINTS
from src.jobs.bronze_streaming import (
    USAGE_EVENTS_BRONZE_PATH,
    USAGE_EVENTS_CHECKPOINT_PATH,
    USAGE_EVENTS_LANDING_GLOB,
)
from src.schemas.bronze_streaming import WATERMARK_DELAY

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT:    {DATA_ROOT}")
print(f"LANDING glob: {USAGE_EVENTS_LANDING_GLOB}")
print(f"BRONZE path:  {USAGE_EVENTS_BRONZE_PATH}")
print(f"CHECKPOINT:   {USAGE_EVENTS_CHECKPOINT_PATH}")
print(f"WATERMARK:    {WATERMARK_DELAY}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("streaming-bronze")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
from src.jobs.bronze_streaming import run_streaming_bronze

# reset_state=True para corrida limpia en desarrollo
result_run1 = run_streaming_bronze(spark, reset_state=True)
result_run1

In [ ]:
import pandas as pd

pd.DataFrame([result_run1])

In [ ]:
df = spark.read.parquet(USAGE_EVENTS_BRONZE_PATH)
df.printSchema()
df.select(
    "event_id", "event_ts", "service", "value", "schema_version",
    "carbon_kg", "genai_tokens", "source_file", "is_late_arrival"
).show(5, truncate=False)

In [ ]:
from src.jobs.bronze_streaming import validate_bronze_streaming

validation = validate_bronze_streaming(spark)
pd.DataFrame([validation])

In [ ]:
# Re-ejecución con checkpoint: no debe duplicar event_id
result_run2 = run_streaming_bronze(spark, reset_state=False)

assert result_run2["written_count"] == result_run2["distinct_event_ids"]
assert result_run2["written_count"] == result_run1["written_count"]
print("Idempotencia OK: re-ejecución sin duplicar event_id.")